# 28 — Pandas Anti-Patterns

What separates junior data scientists from senior ones? It's not knowing more functions,
it's knowing what NOT to do. This notebook covers the 7 deadly sins of Pandas.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 100_000

df = pd.DataFrame({
    'id': range(n),
    'value': np.random.randn(n),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n),
    'status': np.random.choice(['Active', 'Inactive'], n)
})

print(f"Shape: {df.shape}")

## Anti-Pattern 1: Mutating with `inplace=True`

**Why it's bad:**
- It rarely saves memory (Pandas often creates a copy under the hood anyway).
- It breaks method chaining.
- It prevents Pandas from using Copy-on-Write optimizations.

In [ ]:
# ❌ BAD
df_bad = df.copy()
df_bad.drop(columns=['id'], inplace=True)
df_bad.fillna(0, inplace=True)

# ✅ GOOD (Functional style)
df_good = (
    df.copy()
    .drop(columns=['id'])
    .fillna(0)
)

## Anti-Pattern 2: Chained Assignment (SettingWithCopyWarning)

**Why it's bad:**
- `df[df['value'] > 0]['status'] = 'Positive'` modifies a copy, not the original!
- It triggers the SettingWithCopyWarning.

In [ ]:
# ❌ BAD
# df[df['value'] > 0]['status'] = 'Positive'  # Warning!

# ✅ GOOD: Use loc
df_clean = df.copy()
df_clean.loc[df_clean['value'] > 0, 'status'] = 'Positive'

# ✅ BETTER: Use np.where if you are creating a new column
df_clean['is_positive'] = np.where(df_clean['value'] > 0, 1, 0)

## Anti-Pattern 3: Iterating with `iterrows()`

**Why it's bad:**
- It bypasses C-level optimizations and uses Python `for` loops.
- It casts rows to Series, which can change data types.

In [ ]:
import time

small = df.head(5000).copy()

# ❌ BAD
start = time.perf_counter()
results = []
for i, row in small.iterrows():
    results.append(row['value'] * 2 if row['status'] == 'Active' else 0)
print(f"iterrows: {time.perf_counter() - start:.3f}s")

# ✅ GOOD (Vectorized)
start = time.perf_counter()
res_vec = np.where(small['status'] == 'Active', small['value'] * 2, 0)
print(f"vectorized: {time.perf_counter() - start:.4f}s")

## Anti-Pattern 4: Appending Rows in a Loop

**Why it's bad:**
- `pd.concat` inside a loop creates a new DataFrame every time (O(N^2) complexity).

In [ ]:
# ❌ BAD
# result = pd.DataFrame()
# for file in files:
#     df = pd.read_csv(file)
#     result = pd.concat([result, df])  # VERY SLOW

# ✅ GOOD
# dfs = [pd.read_csv(file) for file in files]
# result = pd.concat(dfs, ignore_index=True)  # FAST

## Anti-Pattern 5: Using `apply()` for Math

**Why it's bad:**
- `apply` is just a hidden loop. It does not vectorize operations.

In [ ]:
# ❌ BAD
# df['log_value'] = df['value'].apply(lambda x: np.log(x) if x > 0 else 0)

# ✅ GOOD (Vectorized)
# df['log_value'] = np.where(df['value'] > 0, np.log(df['value']), 0)

## Anti-Pattern 6: Not Using `Categorical` for Low-Cardinality Strings

**Why it's bad:**
- Strings (object dtype) take massive amounts of memory and are slow to compare.

In [ ]:
print("Memory usage (Object):", df['category'].memory_usage(deep=True) // 1024, "KB")

df['category'] = df['category'].astype('category')
print("Memory usage (Category):", df['category'].memory_usage(deep=True) // 1024, "KB")

## Anti-Pattern 7: Complex Multi-Step Filtering Without `query()`

**Why it's bad:**
- Hard to read, lots of bracket repetition, creates intermediate masks in memory.

In [ ]:
target_cat = 'A'

# ❌ BAD
filtered_bad = df[(df['value'] > 0) & (df['category'] == target_cat) & (df['status'] != 'Inactive')]

# ✅ GOOD
filtered_good = df.query("value > 0 and category == @target_cat and status != 'Inactive'")